In [ ]:
# Cell 1: Import required libraries
import os
import tempfile
from typing import List, Dict, Any
import streamlit as st
from dotenv import load_dotenv

# LangChain imports
from langchain.document_loaders import TextLoader, PyPDFLoader, CSVLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.llms import HuggingFacePipeline
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# Load environment variables (if you have any)
load_dotenv()

In [2]:
# Cell 2: Create the core chatbot class
class LightweightRAGChatbot:
    """
    A lightweight RAG chatbot that runs efficiently on CPU
    """
    
    def __init__(self):
        self.embeddings = None
        self.vectorstore = None
        self.chain = None
        self.memory = None
        self.initialize_components()
    
    def initialize_components(self):
        """Initialize embedding model and other components"""
        try:
            # Use a lightweight embedding model
            self.embeddings = HuggingFaceEmbeddings(
                model_name="sentence-transformers/all-MiniLM-L6-v2",
                model_kwargs={'device': 'cpu'},
                encode_kwargs={'normalize_embeddings': True}
            )
            print("✅ Embeddings model loaded successfully!")
        except Exception as e:
            print(f"❌ Error loading embeddings: {e}")
            raise
    
    def create_vectorstore(self, documents):
        """Create a vector store from documents"""
        try:
            # Split documents into chunks
            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=500,
                chunk_overlap=50,
                length_function=len,
                separators=["\n\n", "\n", " ", ""]
            )
            
            chunks = text_splitter.split_documents(documents)
            print(f"✅ Created {len(chunks)} document chunks")
            
            # Create vector store
            self.vectorstore = Chroma.from_documents(
                documents=chunks,
                embedding=self.embeddings,
                persist_directory="./chroma_db"  # Local persistence
            )
            print("✅ Vector store created successfully!")
            return True
            
        except Exception as e:
            print(f"❌ Error creating vectorstore: {e}")
            return False
    
    def create_conversation_chain(self):
        """Create the conversational retrieval chain"""
        if not self.vectorstore:
            print("❌ Vector store not initialized!")
            return False
        
        try:
            # Initialize memory for conversation context
            self.memory = ConversationBufferMemory(
                memory_key="chat_history",
                return_messages=True,
                output_key='answer'
            )
            
            # For CPU-friendly implementation, we'll use a template-based approach
            # without loading a heavy LLM locally
            from langchain.prompts import PromptTemplate
            from langchain.chains import LLMChain
            
            # Define a simple prompt template
            template = """You are a helpful assistant. Use the following context to answer the user's question.
            If you don't know the answer, just say you don't know. Don't try to make up an answer.
            
            Context: {context}
            
            Chat History: {chat_history}
            
            Question: {question}
            
            Answer: Let me help you with that based on the available information."""
            
            prompt = PromptTemplate(
                input_variables=["context", "chat_history", "question"],
                template=template
            )
            
            # For demo purposes, we'll use a mock LLM
            # In production, you can replace this with a lightweight local LLM or API
            from langchain.llms.fake import FakeListLLM
            
            responses = [
                "Based on the context provided, I can help you with that. " +
                "The relevant information shows that this is related to the documents you've provided.",
                "From the available information in your documents, I can tell you that this is an important topic.",
                "According to the context, this is something that is covered in your knowledge base."
            ]
            
            self.llm = FakeListLLM(responses=responses)
            
            # Create the chain
            self.chain = ConversationalRetrievalChain.from_llm(
                llm=self.llm,
                retriever=self.vectorstore.as_retriever(search_kwargs={"k": 3}),
                memory=self.memory,
                combine_docs_chain_kwargs={"prompt": prompt}
            )
            
            print("✅ Conversation chain created successfully!")
            return True
            
        except Exception as e:
            print(f"❌ Error creating conversation chain: {e}")
            return False
    
    def chat(self, message: str) -> str:
        """Process a chat message and return response"""
        if not self.chain:
            return "Chatbot not properly initialized. Please check the setup."
        
        try:
            response = self.chain({"question": message})
            return response['answer']
        except Exception as e:
            return f"Error processing message: {str(e)}"
    
    def clear_memory(self):
        """Clear conversation memory"""
        if self.memory:
            self.memory.clear()
            print("✅ Conversation memory cleared!")

In [3]:
# Cell 3: Document loading utilities
def load_documents_from_text(text_content: str, source: str = "custom_document.txt"):
    """
    Load documents from text content
    """
    from langchain.schema import Document
    
    doc = Document(
        page_content=text_content,
        metadata={"source": source}
    )
    return [doc]

def load_sample_documents():
    """
    Load sample documents for testing
    This creates a small knowledge base about AI and technology
    """
    sample_text = """
    Artificial Intelligence (AI) is the simulation of human intelligence in machines.
    Machine Learning is a subset of AI that enables systems to learn from data.
    Deep Learning uses neural networks with multiple layers for complex pattern recognition.
    
    Natural Language Processing (NLP) helps computers understand human language.
    Transformers are a type of neural network architecture used in modern NLP models.
    BERT and GPT are examples of transformer-based models.
    
    Retrieval-Augmented Generation (RAG) combines information retrieval with text generation.
    RAG systems first retrieve relevant documents then generate responses based on them.
    Vector databases store embeddings for efficient similarity search in RAG applications.
    
    LangChain is a framework for developing applications powered by language models.
    It provides tools for chains, agents, and retrieval strategies.
    Streamlit is a Python library for creating web applications for machine learning.
    
    Python is a popular programming language for AI and data science.
    PyTorch and TensorFlow are deep learning frameworks used for building AI models.
    Hugging Face provides transformers and datasets for NLP tasks.
    """
    
    return load_documents_from_text(sample_text, "sample_knowledge_base.txt")

In [4]:
# Cell 4: Streamlit UI implementation
def create_streamlit_app():
    """
    Create the Streamlit web interface
    """
    st.set_page_config(
        page_title="Lightweight RAG Chatbot",
        page_icon="🤖",
        layout="wide"
    )
    
    st.title("🤖 Context-Aware RAG Chatbot")
    st.markdown("---")
    
    # Initialize session state
    if 'chatbot' not in st.session_state:
        st.session_state.chatbot = LightweightRAGChatbot()
    
    if 'messages' not in st.session_state:
        st.session_state.messages = []
    
    if 'vectorstore_ready' not in st.session_state:
        st.session_state.vectorstore_ready = False
    
    # Sidebar for configuration
    with st.sidebar:
        st.header("📚 Document Management")
        
        # Option to load sample documents
        if st.button("Load Sample Knowledge Base", use_container_width=True):
            with st.spinner("Loading sample documents..."):
                docs = load_sample_documents()
                success = st.session_state.chatbot.create_vectorstore(docs)
                if success:
                    success = st.session_state.chatbot.create_conversation_chain()
                    if success:
                        st.session_state.vectorstore_ready = True
                        st.success("✅ Sample knowledge base loaded!")
                    else:
                        st.error("❌ Failed to create conversation chain")
                else:
                    st.error("❌ Failed to create vector store")
        
        # File upload option
        uploaded_file = st.file_uploader(
            "Upload your own document",
            type=['txt', 'pdf', 'csv'],
            help="Upload a text, PDF, or CSV file to chat with"
        )
        
        if uploaded_file is not None:
            with st.spinner("Processing uploaded document..."):
                try:
                    # Save uploaded file temporarily
                    with tempfile.NamedTemporaryFile(delete=False, suffix=f".{uploaded_file.name.split('.')[-1]}") as tmp_file:
                        tmp_file.write(uploaded_file.getvalue())
                        tmp_path = tmp_file.name
                    
                    # Load based on file type
                    if uploaded_file.type == "text/plain":
                        loader = TextLoader(tmp_path)
                    elif uploaded_file.type == "application/pdf":
                        loader = PyPDFLoader(tmp_path)
                    elif uploaded_file.type == "text/csv":
                        loader = CSVLoader(tmp_path)
                    else:
                        st.error("Unsupported file type")
                        loader = None
                    
                    if loader:
                        documents = loader.load()
                        success = st.session_state.chatbot.create_vectorstore(documents)
                        if success:
                            success = st.session_state.chatbot.create_conversation_chain()
                            if success:
                                st.session_state.vectorstore_ready = True
                                st.success(f"✅ Loaded {len(documents)} documents!")
                            else:
                                st.error("❌ Failed to create conversation chain")
                        else:
                            st.error("❌ Failed to create vector store")
                    
                    # Clean up temp file
                    os.unlink(tmp_path)
                    
                except Exception as e:
                    st.error(f"Error loading file: {str(e)}")
        
        st.markdown("---")
        
        # Clear conversation button
        if st.button("🗑️ Clear Conversation", use_container_width=True):
            st.session_state.messages = []
            if st.session_state.chatbot:
                st.session_state.chatbot.clear_memory()
            st.success("Conversation cleared!")
        
        # Status indicator
        st.markdown("---")
        st.subheader("📊 System Status")
        if st.session_state.vectorstore_ready:
            st.success("✅ Vector store ready")
            st.info("💬 You can now chat with your documents!")
        else:
            st.warning("⚠️ Please load documents first")
    
    # Main chat interface
    st.header("💬 Chat with your documents")
    
    # Display chat messages
    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.markdown(message["content"])
    
    # Chat input
    if prompt := st.chat_input("Ask a question about your documents..."):
        # Check if vectorstore is ready
        if not st.session_state.vectorstore_ready:
            st.warning("Please load documents first using the sidebar!")
        else:
            # Add user message
            st.session_state.messages.append({"role": "user", "content": prompt})
            with st.chat_message("user"):
                st.markdown(prompt)
            
            # Generate response
            with st.chat_message("assistant"):
                with st.spinner("Thinking..."):
                    response = st.session_state.chatbot.chat(prompt)
                    st.markdown(response)
                    
                    # Optional: Show retrieved context in expander
                    if hasattr(st.session_state.chatbot.chain, 'retriever'):
                        with st.expander("📚 Retrieved Context"):
                            docs = st.session_state.chatbot.vectorstore.similarity_search(prompt, k=2)
                            for i, doc in enumerate(docs, 1):
                                st.text(f"Chunk {i}:")
                                st.info(doc.page_content[:200] + "...")
            
            # Add assistant message
            st.session_state.messages.append({"role": "assistant", "content": response})
    
    # Footer
    st.markdown("---")
    st.markdown(
        """
        <div style='text-align: center; color: gray;'>
        Lightweight RAG Chatbot | Built with LangChain + Streamlit | Runs on CPU
        </div>
        """,
        unsafe_allow_html=True
    )

In [ ]:
# Cell 5: Main execution
if __name__ == "__main__":
    # This cell should be run when you want to launch the Streamlit app
    # Note: In a regular Python script, you'd save this as app.py and run: streamlit run app.py
    
    # For demonstration in notebook, we'll create a simple test
    print("🤖 Lightweight RAG Chatbot Setup")
    print("="*50)
    
    # Test the chatbot initialization
    print("\n1. Initializing chatbot...")
    chatbot = LightweightRAGChatbot()
    
    print("\n2. Loading sample documents...")
    docs = load_sample_documents()
    success = chatbot.create_vectorstore(docs)
    
    if success:
        print("\n3. Creating conversation chain...")
        success = chatbot.create_conversation_chain()
        
        if success:
            print("\n4. Testing chatbot with a sample question...")
            response = chatbot.chat("What is RAG?")
            print(f"\nQuestion: What is RAG?")
            print(f"Response: {response}")
            
            print("\n✅ Setup successful! You can now run the Streamlit app.")
            print("\nTo run the Streamlit app, save the code to a file (e.g., app.py)")
            print("and run: streamlit run app.py")
        else:
            print("❌ Failed to create conversation chain")
    else:
        print("❌ Failed to create vector store")